# ARIMA vs ETS: When to Use Which?

ARIMA and ETS (Exponential Smoothing State Space) are the two major families
of time series forecasting methods.  Neither dominates the other — each has
strengths and weaknesses:

| | ARIMA | ETS |
|---|---|---|
| **Approach** | Model autocorrelation structure | Decompose into level, trend, season |
| **Stationarity** | Requires differencing to achieve it | Does not require stationarity |
| **Seasonality** | Additive only (use log for multiplicative) | Handles multiplicative natively |
| **Interpretability** | Coefficients less intuitive | Components (level, trend, season) are meaningful |
| **Complex autocorrelation** | Can model with AR/MA terms | Limited to exponential smoothing patterns |
| **Prediction intervals** | Yes (from state-space form) | Yes (from state-space form) |
| **Automatic selection** | `auto_arima` (AIC/BIC search) | `ets()` in R; manual in Python |

This notebook runs a **head-to-head comparison** on multiple datasets to
see which performs better in practice.

> **Practical advice** (FPP3 Section 9.10): "In practice, try both and
> pick the one with better out-of-sample accuracy."

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Conceptual Differences

### ARIMA thinks in terms of autocorrelation

ARIMA models the **correlation structure** of the (differenced) series.
AR terms capture how past values influence the present; MA terms capture
how past shocks propagate.  The "I" part handles non-stationarity through
differencing.

### ETS thinks in terms of components

ETS decomposes the series into **error, trend, and seasonal** components,
with each component updated via exponential smoothing equations.  The
smoothing parameters ($\alpha$, $\beta$, $\gamma$) control how quickly
each component adapts to new data.

### Overlap

Some models are equivalent:
- Simple Exponential Smoothing $\approx$ ARIMA(0,1,1)
- Holt's linear method $\approx$ ARIMA(0,2,2)
- Damped trend method $\approx$ ARIMA(1,1,2)

But many models in each family have **no equivalent** in the other.

---
## 2. Dataset 1: Airline Passengers

Classic seasonal data with trend and multiplicative seasonality.

In [ ]:
airline = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    index_col="Month",
    parse_dates=True,
)
airline.index.freq = "MS"
airline.columns = ["Passengers"]

TRAIN_SIZE = 120
air_train = airline.iloc[:TRAIN_SIZE]
air_test = airline.iloc[TRAIN_SIZE:]
air_h = len(air_test)

print(f"Train: {len(air_train)} months, Test: {len(air_test)} months")
print(f"Forecast horizon: {air_h} months")

### Fit Best ARIMA

In [ ]:
# auto_arima to find optimal SARIMA
arima_model = auto_arima(
    air_train["Passengers"],
    seasonal=True,
    m=12,
    stepwise=True,
    trace=True,
    suppress_warnings=True,
)

print(f"\nBest ARIMA: {arima_model.order} x {arima_model.seasonal_order}")
print(f"AIC: {arima_model.aic():.2f}")

In [ ]:
# ARIMA forecast
arima_fc, arima_ci = arima_model.predict(
    n_periods=air_h, return_conf_int=True, alpha=0.05
)

### Fit Best ETS

In [ ]:
# Try multiple ETS configurations and pick the best by AIC
ets_configs = [
    {"trend": "add", "seasonal": "add", "seasonal_periods": 12},
    {"trend": "add", "seasonal": "mul", "seasonal_periods": 12},
    {"trend": "mul", "seasonal": "mul", "seasonal_periods": 12},
    {"trend": "add", "seasonal": "add", "seasonal_periods": 12, "damped_trend": True},
    {"trend": "add", "seasonal": "mul", "seasonal_periods": 12, "damped_trend": True},
    {"trend": "mul", "seasonal": "mul", "seasonal_periods": 12, "damped_trend": True},
]

best_ets_aic = np.inf
best_ets_result = None
best_ets_desc = ""

for config in ets_configs:
    damped = config.pop("damped_trend", False)
    try:
        mod = ExponentialSmoothing(
            air_train["Passengers"],
            **config,
            damped_trend=damped,
        ).fit(optimized=True)
        desc = f"ETS(trend={config['trend']}, seasonal={config['seasonal']}, damped={damped})"
        print(f"  {desc}: AIC={mod.aic:.2f}")
        if mod.aic < best_ets_aic:
            best_ets_aic = mod.aic
            best_ets_result = mod
            best_ets_desc = desc
    except Exception as e:
        pass

print(f"\nBest ETS: {best_ets_desc}")
print(f"AIC: {best_ets_aic:.2f}")

In [ ]:
# ETS forecast
ets_fc = best_ets_result.forecast(air_h)

### Compare on Airline Passengers

In [ ]:
actual = air_test["Passengers"].values

def compute_metrics(actual, predicted, name):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    return {"Model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE": round(mape, 2)}


# Seasonal naive for baseline
snaive_fc = np.tile(air_train["Passengers"].values[-12:],
                     int(np.ceil(air_h / 12)))[:air_h]

metrics = pd.DataFrame([
    compute_metrics(actual, arima_fc, f"SARIMA {arima_model.order}x{arima_model.seasonal_order}"),
    compute_metrics(actual, ets_fc.values, best_ets_desc),
    compute_metrics(actual, snaive_fc, "Seasonal Naive"),
]).sort_values("MAE").reset_index(drop=True)

print("=== Airline Passengers — Accuracy Comparison ===")
print(metrics.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(air_train["Passengers"].iloc[-36:], label="Train (last 3 yr)",
        color="black", alpha=0.4)
ax.plot(air_test["Passengers"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(air_test.index, arima_fc, label="ARIMA", color="tab:red", linestyle="--")
ax.plot(air_test.index, ets_fc.values, label="ETS", color="tab:green", linestyle="--")
ax.plot(air_test.index, snaive_fc, label="Seasonal Naive",
        color="tab:purple", linestyle=":")
ax.fill_between(air_test.index, arima_ci[:, 0], arima_ci[:, 1],
                color="tab:red", alpha=0.08, label="ARIMA 95% CI")
ax.set_title("Airline Passengers — ARIMA vs ETS vs Seasonal Naive")
ax.set_ylabel("Thousands of Passengers")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Dataset 2: Monthly Milk Production

Another classic seasonal dataset — monthly milk production per cow.

In [ ]:
milk = pd.read_csv(
    DATA_DIR / "monthly_milk_production.csv",
    index_col=0,
    parse_dates=True,
)
milk.columns = ["Production"]
milk.index.freq = "MS"

print(f"Shape: {milk.shape}")
print(f"Range: {milk.index[0].date()} to {milk.index[-1].date()}")

fig, ax = plt.subplots()
ax.plot(milk["Production"])
ax.set_title("Monthly Milk Production per Cow")
ax.set_ylabel("Pounds")
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
milk_train = milk.iloc[:-24]
milk_test = milk.iloc[-24:]
milk_h = len(milk_test)

print(f"Train: {len(milk_train)}, Test: {len(milk_test)}")

In [ ]:
# Fit ARIMA
milk_arima = auto_arima(
    milk_train["Production"],
    seasonal=True,
    m=12,
    stepwise=True,
    suppress_warnings=True,
)
print(f"ARIMA: {milk_arima.order} x {milk_arima.seasonal_order}")

milk_arima_fc, milk_arima_ci = milk_arima.predict(
    n_periods=milk_h, return_conf_int=True, alpha=0.05
)

In [ ]:
# Fit ETS — try additive and multiplicative seasonal
best_milk_ets = None
best_milk_aic = np.inf
best_milk_desc = ""

for trend in ["add", "mul"]:
    for seasonal in ["add", "mul"]:
        for damped in [False, True]:
            try:
                mod = ExponentialSmoothing(
                    milk_train["Production"],
                    trend=trend,
                    seasonal=seasonal,
                    seasonal_periods=12,
                    damped_trend=damped,
                ).fit(optimized=True)
                desc = f"ETS(T={trend},S={seasonal},D={damped})"
                if mod.aic < best_milk_aic:
                    best_milk_aic = mod.aic
                    best_milk_ets = mod
                    best_milk_desc = desc
            except Exception:
                pass

print(f"Best ETS: {best_milk_desc}, AIC={best_milk_aic:.2f}")
milk_ets_fc = best_milk_ets.forecast(milk_h)

In [ ]:
# Compare on milk data
milk_actual = milk_test["Production"].values
milk_snaive = np.tile(milk_train["Production"].values[-12:],
                       int(np.ceil(milk_h / 12)))[:milk_h]

milk_metrics = pd.DataFrame([
    compute_metrics(milk_actual, milk_arima_fc,
                    f"SARIMA {milk_arima.order}x{milk_arima.seasonal_order}"),
    compute_metrics(milk_actual, milk_ets_fc.values, best_milk_desc),
    compute_metrics(milk_actual, milk_snaive, "Seasonal Naive"),
]).sort_values("MAE").reset_index(drop=True)

print("=== Milk Production — Accuracy Comparison ===")
print(milk_metrics.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(milk_train["Production"].iloc[-36:], label="Train (last 3 yr)",
        color="black", alpha=0.4)
ax.plot(milk_test["Production"], label="Actual", color="tab:blue", linewidth=2)
ax.plot(milk_test.index, milk_arima_fc, label="ARIMA", color="tab:red", linestyle="--")
ax.plot(milk_test.index, milk_ets_fc.values, label="ETS", color="tab:green", linestyle="--")
ax.plot(milk_test.index, milk_snaive, label="Seasonal Naive",
        color="tab:purple", linestyle=":")
ax.set_title("Milk Production — ARIMA vs ETS")
ax.set_ylabel("Pounds")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 4. Prediction Interval Comparison

Both ARIMA and ETS produce prediction intervals.  Let us compare their
width and coverage on the airline passengers data.

In [ ]:
# ARIMA prediction interval width
arima_width = arima_ci[:, 1] - arima_ci[:, 0]

# ETS prediction interval (simulate to get intervals)
# ExponentialSmoothing in statsmodels supports simulate for PI
ets_sim = best_ets_result.simulate(
    air_h, repetitions=1000, anchor="end",
    error="mul" if "mul" in best_ets_desc else "add",
)
ets_lower = ets_sim.quantile(0.025, axis=1)
ets_upper = ets_sim.quantile(0.975, axis=1)
ets_width = ets_upper.values - ets_lower.values

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, air_h + 1), arima_width, label="ARIMA 95% CI width", marker="o", markersize=4)
ax.plot(range(1, air_h + 1), ets_width, label="ETS 95% CI width", marker="s", markersize=4)
ax.set_xlabel("Forecast Horizon (months)")
ax.set_ylabel("Interval Width")
ax.set_title("Prediction Interval Width: ARIMA vs ETS")
ax.legend()
plt.tight_layout()
plt.show()

print("Both intervals widen with horizon.  Differences in width reflect")
print("different model assumptions about error propagation.")

In [ ]:
# Coverage: what fraction of test points fall within the 95% CI?
arima_coverage = np.mean((actual >= arima_ci[:, 0]) & (actual <= arima_ci[:, 1]))
ets_coverage = np.mean((actual >= ets_lower.values) & (actual <= ets_upper.values))

print(f"ARIMA 95% CI coverage: {arima_coverage:.1%}")
print(f"ETS   95% CI coverage: {ets_coverage:.1%}")
print(f"\nIdeal coverage is 95%. Both models may over- or under-cover")
print(f"depending on whether their assumptions are satisfied.")

---
## 5. When to Prefer ARIMA

- **Non-seasonal data** with complex autocorrelation structure
- Data where the ACF/PACF suggest specific AR and MA terms
- When you need to model the **correlation between consecutive forecast errors**
- Data that becomes stationary after differencing

## 6. When to Prefer ETS

- **Seasonal data with multiplicative seasonality** — ETS handles this natively
- When you want **interpretable components** (level, trend, season)
- Short time series where component decomposition is more stable
- When the smoothing parameter interpretation ($\alpha$, $\beta$, $\gamma$)
  provides useful domain insight

---
## 7. Overall Results Summary

In [ ]:
print("=" * 60)
print("DATASET 1: Airline Passengers (trend + multiplicative season)")
print("=" * 60)
print(metrics.to_string(index=False))
print()
print("=" * 60)
print("DATASET 2: Milk Production (trend + seasonal)")
print("=" * 60)
print(milk_metrics.to_string(index=False))
print()
print("Key takeaways:")
print("  - Both ARIMA and ETS outperform the seasonal naive baseline.")
print("  - The winner varies by dataset — no single method dominates.")
print("  - In practice: fit both, evaluate on a hold-out set, pick the best.")

---
## 8. A Practical Workflow

When faced with a new forecasting problem:

1. **Explore** the data — plot, check for trend, seasonality, outliers
2. **Split** into train/test (hold out the last $h$ periods)
3. **Fit baselines**: seasonal naive (if seasonal) or naive (if not)
4. **Fit ARIMA**: use `auto_arima` with appropriate `seasonal` and `m`
5. **Fit ETS**: try multiple (trend, seasonal) configurations, pick best AIC
6. **Evaluate** on the test set using MAE, RMSE, MAPE
7. **Check residuals** for the best model — are they white noise?
8. **Select** the model with the best test-set accuracy
9. **Refit** on the full dataset and produce the final forecast

In [ ]:
def compare_arima_ets(train_series, test_series, m=12, name="Dataset"):
    """Run ARIMA vs ETS comparison and return results."""
    h = len(test_series)
    actual = test_series.values

    # ARIMA
    arima_mod = auto_arima(
        train_series, seasonal=True, m=m, stepwise=True, suppress_warnings=True
    )
    arima_pred = arima_mod.predict(n_periods=h)

    # ETS — search over configurations
    best_aic = np.inf
    best_ets = None
    for trend in ["add", "mul"]:
        for seasonal in ["add", "mul"]:
            for damped in [False, True]:
                try:
                    mod = ExponentialSmoothing(
                        train_series, trend=trend, seasonal=seasonal,
                        seasonal_periods=m, damped_trend=damped,
                    ).fit(optimized=True)
                    if mod.aic < best_aic:
                        best_aic = mod.aic
                        best_ets = mod
                except Exception:
                    pass

    ets_pred = best_ets.forecast(h).values

    # Seasonal naive
    snaive = np.tile(train_series.values[-m:], int(np.ceil(h / m)))[:h]

    results = pd.DataFrame([
        compute_metrics(actual, arima_pred,
                        f"ARIMA {arima_mod.order}x{arima_mod.seasonal_order}"),
        compute_metrics(actual, ets_pred, "Best ETS"),
        compute_metrics(actual, snaive, "Seasonal Naive"),
    ]).sort_values("MAE").reset_index(drop=True)

    print(f"\n=== {name} ===")
    print(results.to_string(index=False))
    return results


print("Reusable comparison function defined.")

---
## Summary

- **ARIMA and ETS** are the two major families of statistical time series
  forecasting.  Neither universally dominates the other.
- ARIMA excels at modelling **complex autocorrelation** structures and
  works naturally with differencing for non-stationary data.
- ETS excels at handling **multiplicative seasonality** natively and
  provides **interpretable components** (level, trend, season).
- Some models overlap (SES $\approx$ ARIMA(0,1,1)), but many are unique
  to each family.
- Both produce **prediction intervals** from their state-space forms.
- **Practical approach**: fit both with automatic selection (`auto_arima`,
  ETS grid search), evaluate on a hold-out test set, and choose the model
  with the best out-of-sample accuracy.
- Both should be evaluated against a **simple benchmark** (seasonal naive
  for seasonal data, naive for non-seasonal data).

In [ ]:
print("This concludes the ARIMA chapter.  You now have a complete toolkit")
print("for univariate time series forecasting with both ETS and ARIMA.")
print("\nNext steps: regression with ARIMA errors (ARIMAX/SARIMAX), or")
print("moving to machine learning approaches for time series.")